# Dual-IMU feature reliability validation

This notebook demonstrates windows that should be considered reliable vs unreliable under the new feature reliability logic.

In [ ]:
import numpy as np
from features.extract import _cross_corr_lag_ms, _signal_stats_ok, MIN_DENOM_ENERGY
from features.signal_stats import safe_ratio

In [ ]:
# Reliable window: clear periodic content with known lag
fs = 50
t = np.arange(0, 2, 1/fs)
bike = np.sin(2*np.pi*2.0*t) + 0.1*np.random.randn(len(t))
rider = np.sin(2*np.pi*2.0*(t-0.08)) + 0.1*np.random.randn(len(t))
lag_ms, lag_conf, lag_reasons = _cross_corr_lag_ms(bike, rider, dt_ms=1000/fs)
q_bike, q_bike_reasons = _signal_stats_ok(bike)
q_rider, q_rider_reasons = _signal_stats_ok(rider)
print({'lag_ms': lag_ms, 'lag_conf': lag_conf, 'lag_reasons': lag_reasons})
print({'bike_quality': q_bike, 'bike_reasons': q_bike_reasons})
print({'rider_quality': q_rider, 'rider_reasons': q_rider_reasons})

In [ ]:
# Unreliable window: low-variance / flat signals
bike_flat = np.full(100, 0.001)
rider_flat = np.full(100, 0.0012)
lag_ms, lag_conf, lag_reasons = _cross_corr_lag_ms(bike_flat, rider_flat, dt_ms=20.0)
q_bike, q_bike_reasons = _signal_stats_ok(bike_flat)
q_rider, q_rider_reasons = _signal_stats_ok(rider_flat)
print({'lag_ms': lag_ms, 'lag_conf': lag_conf, 'lag_reasons': lag_reasons})
print({'bike_quality': q_bike, 'bike_reasons': q_bike_reasons})
print({'rider_quality': q_rider, 'rider_reasons': q_rider_reasons})

In [ ]:
# Denominator guard behaviour for energy-ratio style features
num = 0.12
den_good = 0.25
den_tiny = 1e-7
print('ratio_good', safe_ratio(num, den_good, eps=MIN_DENOM_ENERGY))
print('ratio_tiny', safe_ratio(num, den_tiny, eps=MIN_DENOM_ENERGY))